# SessionRerank — full ToolBench benchmark on Colab T4

Reproduces every metric in `results/metrics.json` using ToolBench's released paper-faithful encoder `ToolBench/ToolBench_IR_bert_based_uncased`. Expected runtime on a single T4 GPU: **~3-4 minutes**.

Steps:
1. Install deps & clone repo
2. Download ToolBench (HuggingFace mirror)
3. Run full benchmark (BM25 / Dense / ToolRerank / SessionRerank)
4. Plot publication-ready figures
5. Display results

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/joyjeni/session-aware-toolbench-rerank.git
%cd session-aware-toolbench-rerank
import os
REPO_DIR = os.getcwd()
os.environ['REPO_DIR'] = REPO_DIR
print(f'[repo] REPO_DIR={REPO_DIR}')
!pip install -q -r requirements.txt

In [ ]:
import sys, os
sys.path.insert(0, 'src')
os.environ['PYTHONPATH'] = 'src'
!python -m sessionrerank.data.download

## Run benchmark with paper-faithful encoder on T4

In [ ]:
!python -m sessionrerank.eval.run_all \
    --device cuda \
    --encoder ToolBench/ToolBench_IR_bert_based_uncased \
    --fit-sessions 8000 \
    --out results/metrics_t4.json

In [ ]:
!python -m sessionrerank.viz.make_charts

In [ ]:
import json
from IPython.display import display, Image, Markdown
m = json.loads(open('results/metrics_t4.json').read())
rows = []
for split, blk in m['splits'].items():
    for method, mtr in blk['regimes']['session'].items():
        rows.append((split, method, mtr['NDCG@5'], mtr['Hit@5']))
import pandas as pd
df = pd.DataFrame(rows, columns=['Split','Method','NDCG@5','Hit@5'])
display(df.pivot(index='Split', columns='Method', values='NDCG@5').round(3))
for fn in ('fig1_ndcg5_session','fig3_ndcg_atK_g2inst','fig4_relgain_vs_dense','fig5_heatmap_ndcg5'):
    display(Image(f'results/figures/{fn}.png'))